# Fake Review Detection – Dataset EDA & Model Training

This notebook:
1. Inspects both datasets (Op-Spam & Yelp)
2. Checks class balance and text-length distributions
3. Runs the full train → evaluate pipeline
4. Displays evaluation metrics and example explanations

In [ ]:
import sys, pathlib
# Ensure project root is on path
ROOT = pathlib.Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
import logging
logging.basicConfig(level=logging.INFO)
print('Root:', ROOT)

## 1. Load Both Datasets

In [ ]:
import os; os.chdir(ROOT)  # run from project root
from ml.src.data_loader import load_opspam, load_yelp, load_all

op_df = load_opspam()
yelp_df = load_yelp()
all_df = load_all()
print('Op-Spam shape:', op_df.shape)
print('Yelp shape:   ', yelp_df.shape)
print('Combined:     ', all_df.shape)
op_df.head(3)

## 2. Class Balance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, df) in zip(axes, [('Op-Spam', op_df), ('Yelp', yelp_df)]):
    counts = df['label'].value_counts().rename({0: 'Genuine', 1: 'Deceptive'})
    counts.plot(kind='bar', ax=ax, color=['#4CAF50', '#F44336'], rot=0)
    ax.set_title(f'{name} – Label Distribution')
    ax.set_ylabel('Count')
    for p in ax.patches:
        ax.annotate(str(int(p.get_height())), (p.get_x()+0.1, p.get_height()+10))
plt.tight_layout()
plt.savefig('docs/test_reports/class_balance.png', dpi=100)
plt.show()

## 3. Text Length Distribution

In [ ]:
op_df['word_count'] = op_df['text'].str.split().str.len()
yelp_df['word_count'] = yelp_df['text'].str.split().str.len()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (name, df) in zip(axes, [('Op-Spam', op_df), ('Yelp', yelp_df)]):
    for lbl, grp in df.groupby('label'):
        grp['word_count'].clip(0, 500).plot.hist(
            bins=40, alpha=0.6, ax=ax,
            label='Genuine' if lbl==0 else 'Deceptive'
        )
    ax.set_title(f'{name} – Word Count Distribution')
    ax.set_xlabel('Word Count')
    ax.legend()
plt.tight_layout()
plt.savefig('docs/test_reports/word_length_dist.png', dpi=100)
plt.show()

print('Op-Spam stats:\n', op_df.groupby('label')['word_count'].describe())
print('\nYelp stats:\n', yelp_df.groupby('label')['word_count'].describe())

## 4. Cross-Dataset Split Strategy

In [ ]:
print(all_df.groupby(['split','source','label']).size().reset_index(name='count').to_string(index=False))
print()
print('Strategy:')
print('  Op-Spam folds 1-4 → TRAIN (balanced 50/50)')
print('  Op-Spam fold5    → IN-DOMAIN TEST')
print('  Yelp (all)       → CROSS-DATASET TEST  (class_weight=balanced handles imbalance)')

## 5. Train All Models

In [ ]:
from ml.src.train import train
fitted_models, extractor = train()  # fits + saves to models/
print('Trained:', list(fitted_models.keys()))

## 6. Evaluate All Models

In [ ]:
from ml.src.evaluate import evaluate
results = evaluate()
import pandas as pd
summary = pd.DataFrame([{k:v for k,v in r.items() if k!='class_report'} for r in results])
summary

## 7. Example Explanation

In [ ]:
from ml.src.explainability import explain
import json

# A clearly promotional review
fake_review = (
    'Best hotel EVER!!! Absolutely AMAZING, the BEST in the whole city!! '
    'Perfect in every single way, I cannot recommend it enough!!!'
)
genuine_review = (
    'Stayed here for 3 nights in October. The room on the 14th floor had a '
    'nice view but the air conditioning was noisy and kept me up. Breakfast '
    'at the buffet was decent – eggs were fresh but the pastries were stale.'
)

print('=== Promotional review ===')
print(json.dumps(explain(fake_review, model_name='lr'), indent=2))
print('\n=== Genuine-style review ===')
print(json.dumps(explain(genuine_review, model_name='lr'), indent=2))